In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
from matplotlib.ticker import FuncFormatter
import warnings
warnings.filterwarnings('ignore')
import re

In [3]:
df = pd.read_csv('../SRA_no_16S.csv', index_col=False)
total_bytes = df['Bytes'].sum()
print(f"Общий объем данных: {total_bytes/1e12:.4f} TB")

Общий объем данных: 3.0792 TB


### Чистим от 16S rRNA было 2807 - стало 1707

In [ ]:
deleted_rows = df[df['target_gene (exp)'] == '16S rRNA']
df = df[df['target_gene (exp)'] != '16S rRNA']
#df
deleted_rows

### Убираем запуски с аномально большим количеством баз (больше 30 гигабаз и весят по 50-150 гигабайт) (пока не убираем)

In [141]:
#df = df[df['Bases'].apply(lambda x: x<30000000000)]
#df

### Удаляем хозяев мышь и выдру

In [ ]:
df = df[(df['HOST'] != 'Mus musculus') & (df['HOST'] !='southern_sea_otter')]
df

### Удаляем проект PRJNA1214429 с 311 архивами, в которых все прочтения бактериальные и PRJNA665586 (4 архива) просто мазок из пещеры - вирусных прочтений нет.

In [8]:
df = df[(df['BioProject'] != 'PRJNA1214429') & (df['BioProject'] != 'PRJNA665586')]
df

,Run,Assay Type,AvgSpotLen,Bases,BioProject,BioSample,Bytes,Center Name,Consent,DATASTORE filetype,...,identified_by,label,material,sample_ID,Derived_from,Host_Diet,metagenome_source,nat_host,organism,rel_to_oxygen
0,SRR868695,WGS,168,32700916,PRJNA203532,SAMN02147246,24426068,UNIVERSITY OF TURKU,public,"fastq,run.zq,sra",...,NaN,bat-fin-2018,feces,NaN,NaN,insectivorous,NaN,NaN,NaN,NaN
1,SRR975462,OTHER,200,1112399600,PRJNA218570,SAMN02351598,742222275,INSTITUTE OF MILITARY VETERINARY,public,"fastq,sra",...,NaN,NaN,feces,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,SRR6202122,WGS,202,18066072,PRJNA415003,SAMN07811889,9331475,BGI,public,"fastq,run.zq,sra",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,SRR6202123,WGS,202,5645294,PRJNA415003,SAMN07811890,3525641,BGI,public,"fastq,run.zq,sra",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,SRR6202124,WGS,202,6025458,PRJNA415003,SAMN07811891,3733079,BGI,public,"fastq,run.zq,sra",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1617,ERR15951205,WGS,258,37874548118,PRJEB103765,SAMEA120589895,11473741804,QUADRAM INSTITUTE BIOSCIENCE,public,"sra,run.zq,fastq",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1618,ERR15951206,WGS,275,57310186414,PRJEB103765,SAMEA120589896,17777169612,QUADRAM INSTITUTE BIOSCIENCE,public,"run.zq,fastq,sra",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1619,ERR15951207,WGS,279,67460067788,PRJEB103765,SAMEA120589897,20797734971,QUADRAM INSTITUTE BIOSCIENCE,public,"run.zq,fastq,sra",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1620,SRR37069663,WGS,7864,1978643193,PRJNA1417558,SAMN54999376,622989701,JILIN AGRICULTURAL UNIVERSITY,public,"bam,sra",...,Jilin Provincial Key Laboratory of Animal Reso...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Удаляем проекты, где организм бактерии

In [13]:
pattern = r'(Escherichia coli|Lactobacillus|Lactococcus sp\.|Escherichia marmotae|Clostridium sp\.|Enterobacteriaceae bacterium|Streptomyces sp\.|Staphylococcus nepalensis)'

# organism filter
mask_org = ~df['Organism'].str.contains(pattern, case=False, na=False, regex=True)

# source_name filter
mask_source = ~df['source_name'].str.contains(r'(lung|blood)', case=False, na=False)

# tissue filter
mask_tissue = ~df['tissue'].str.contains(r'(lung|blood)', case=False, na=False)

# combine all filters
mask = mask_org & mask_source & mask_tissue

filtered_df = df[mask]
removed = df[~mask]

print(f"Удалено {len(removed)} строк:")

df = filtered_df

Удалено 19 строк:


In [14]:
df

,Run,Assay Type,AvgSpotLen,Bases,BioProject,BioSample,Bytes,Center Name,Consent,DATASTORE filetype,...,env_package,experimental_factor,feature,identified_by,label,material,sample_ID,Host_Diet,organism,rel_to_oxygen
0,SRR868695,WGS,168,32700916,PRJNA203532,SAMN02147246,24426068,UNIVERSITY OF TURKU,public,"fastq,run.zq,sra",...,MIGS/MIMS/MIMARKS.host-associated,NaN,sea coast,NaN,bat-fin-2018,feces,NaN,insectivorous,NaN,NaN
1,SRR975462,OTHER,200,1112399600,PRJNA218570,SAMN02351598,742222275,INSTITUTE OF MILITARY VETERINARY,public,"fastq,sra",...,MIGS/MIMS/MIMARKS.host-associated,NaN,feces,NaN,NaN,feces,NaN,NaN,NaN,NaN
2,SRR6202122,WGS,202,18066072,PRJNA415003,SAMN07811889,9331475,BGI,public,"fastq,run.zq,sra",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,SRR6202123,WGS,202,5645294,PRJNA415003,SAMN07811890,3525641,BGI,public,"fastq,run.zq,sra",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,SRR6202124,WGS,202,6025458,PRJNA415003,SAMN07811891,3733079,BGI,public,"fastq,run.zq,sra",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1617,ERR15951205,WGS,258,37874548118,PRJEB103765,SAMEA120589895,11473741804,QUADRAM INSTITUTE BIOSCIENCE,public,"sra,run.zq,fastq",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1618,ERR15951206,WGS,275,57310186414,PRJEB103765,SAMEA120589896,17777169612,QUADRAM INSTITUTE BIOSCIENCE,public,"run.zq,fastq,sra",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1619,ERR15951207,WGS,279,67460067788,PRJEB103765,SAMEA120589897,20797734971,QUADRAM INSTITUTE BIOSCIENCE,public,"run.zq,fastq,sra",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1620,SRR37069663,WGS,7864,1978643193,PRJNA1417558,SAMN54999376,622989701,JILIN AGRICULTURAL UNIVERSITY,public,"bam,sra",...,NaN,NaN,NaN,Jilin Provincial Key Laboratory of Animal Reso...,NaN,NaN,NaN,NaN,NaN,NaN


### Убираем пустые столбцы. В итоге остается 1189 архивов

In [18]:
df = df.dropna(axis=1, how='all')
df.to_csv('../sra_filtered_no_16S.csv', index=False)

In [19]:
df

,Run,Assay Type,AvgSpotLen,Bases,BioProject,BioSample,Bytes,Center Name,Consent,DATASTORE filetype,...,env_package,experimental_factor,feature,identified_by,label,material,sample_ID,Host_Diet,organism,rel_to_oxygen
0,SRR868695,WGS,168,32700916,PRJNA203532,SAMN02147246,24426068,UNIVERSITY OF TURKU,public,"fastq,run.zq,sra",...,MIGS/MIMS/MIMARKS.host-associated,NaN,sea coast,NaN,bat-fin-2018,feces,NaN,insectivorous,NaN,NaN
1,SRR975462,OTHER,200,1112399600,PRJNA218570,SAMN02351598,742222275,INSTITUTE OF MILITARY VETERINARY,public,"fastq,sra",...,MIGS/MIMS/MIMARKS.host-associated,NaN,feces,NaN,NaN,feces,NaN,NaN,NaN,NaN
2,SRR6202122,WGS,202,18066072,PRJNA415003,SAMN07811889,9331475,BGI,public,"fastq,run.zq,sra",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,SRR6202123,WGS,202,5645294,PRJNA415003,SAMN07811890,3525641,BGI,public,"fastq,run.zq,sra",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,SRR6202124,WGS,202,6025458,PRJNA415003,SAMN07811891,3733079,BGI,public,"fastq,run.zq,sra",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1617,ERR15951205,WGS,258,37874548118,PRJEB103765,SAMEA120589895,11473741804,QUADRAM INSTITUTE BIOSCIENCE,public,"sra,run.zq,fastq",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1618,ERR15951206,WGS,275,57310186414,PRJEB103765,SAMEA120589896,17777169612,QUADRAM INSTITUTE BIOSCIENCE,public,"run.zq,fastq,sra",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1619,ERR15951207,WGS,279,67460067788,PRJEB103765,SAMEA120589897,20797734971,QUADRAM INSTITUTE BIOSCIENCE,public,"run.zq,fastq,sra",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1620,SRR37069663,WGS,7864,1978643193,PRJNA1417558,SAMN54999376,622989701,JILIN AGRICULTURAL UNIVERSITY,public,"bam,sra",...,NaN,NaN,NaN,Jilin Provincial Key Laboratory of Animal Reso...,NaN,NaN,NaN,NaN,NaN,NaN


In [146]:
#df['tissue'].value_counts()
#df['HOST'].value_counts()
#df['Organism'].value_counts()
#df.loc[df['BioProject'] == 'PRJNA665586']
#df.loc[df['HOST'] == 'Environment']
#df.loc[df['isolation_source'] == 'forest']

### Данные весят 2.58 терабайт

In [20]:
df = pd.read_csv('../sra_filtered_no_16S.csv')
total_bytes = df['Bytes'].sum()
print(f"Общий объем данных: {total_bytes/1e12:.4f} TB")

Общий объем данных: 2.8610 TB


In [21]:
print("\nБольшие файлы:")
top_large = df.nlargest(10, 'Bytes')[['Run', 'Bytes', 'Assay Type', 'Organism']].copy()
top_large['Size_GB'] = top_large['Bytes'] / 1e9
top_large['Size_TB'] = top_large['Bytes'] / 1e12
print(top_large[['Run', 'Size_GB', 'Size_TB', 'Assay Type', 'Organism']])


Большие файлы:
              Run     Size_GB   Size_TB Assay Type              Organism
919   SRR23314745  161.519252  0.161519    RNA-Seq        bat metagenome
920   SRR23314746  123.813281  0.123813    RNA-Seq        bat metagenome
917   SRR23314743  100.071383  0.100071    RNA-Seq        bat metagenome
1150  ERR15951201   67.632926  0.067633        WGS    bat gut metagenome
921   SRR23314747   61.614715  0.061615    RNA-Seq        bat metagenome
925   SRR23314751   57.810831  0.057811    RNA-Seq        bat metagenome
924   SRR23314750   53.888778  0.053889    RNA-Seq        bat metagenome
923   SRR23314749   34.270534  0.034271    RNA-Seq        bat metagenome
495   SRR14860617   33.661649  0.033662        WGS        gut metagenome
842   SRR18679138   32.391888  0.032392    RNA-Seq  Alphacoronavirus sp.


In [22]:
categorical_cols = ['Assay Type', 'LibraryLayout', 'LibrarySource', 'Platform', 'geo_loc_name_country', 'geo_loc_name_country_continent', 'sample_type']
for col in categorical_cols:
    if col in df.columns:
        print(f"\n{col}:")
        print(f"  Количество уникальных значений: {df[col].nunique()}")
        print(f"  Топ-10 значений: {df[col].value_counts().head(10).to_dict()}")


Assay Type:
  Количество уникальных значений: 3
  Топ-10 значений: {'WGS': 693, 'RNA-Seq': 355, 'OTHER': 110}

LibraryLayout:
  Количество уникальных значений: 2
  Топ-10 значений: {'PAIRED': 926, 'SINGLE': 232}

LibrarySource:
  Количество уникальных значений: 6
  Топ-10 значений: {'METAGENOMIC': 749, 'METATRANSCRIPTOMIC': 214, 'GENOMIC': 116, 'VIRAL RNA': 47, 'TRANSCRIPTOMIC': 28, 'OTHER': 4}

Platform:
  Количество уникальных значений: 7
  Топ-10 значений: {'ILLUMINA': 974, 'OXFORD_NANOPORE': 157, 'DNBSEQ': 21, 'CAPILLARY': 3, 'ION_TORRENT': 1, 'GENEMIND': 1, 'PACBIO_SMRT': 1}

geo_loc_name_country:
  Количество уникальных значений: 39
  Топ-10 значений: {'China': 177, 'Brazil': 172, 'Panama': 101, 'Spain': 99, 'Switzerland': 70, 'Singapore': 62, 'United Kingdom': 47, 'USA': 43, 'uncalculated': 41, 'Australia': 38}

geo_loc_name_country_continent:
  Количество уникальных значений: 7
  Топ-10 значений: {'Asia': 314, 'Europe': 294, 'South America': 173, 'North America': 167, 'uncalcu